# Intro til Machine Learning — 2: Neurale netværk

I notebook 1 fandt gradient descent den bedste **linje**. Men verden er sjældent en linje:
nogle mønstre er krøllede, buede og indviklede. Løsningen er at stable mange små
linje-byggeklodser oven på hinanden med lidt "krølning" imellem — og så har man et
**neuralt netværk**, den model der har drevet AI-revolutionen.

I denne notebook:
1. **Neuronen** — netværkets mindste byggesten (afsnit 4)
2. **`nn.Module`** — netværk skrives som *klasser* (jeres nye klasse-viden i aktion!) (afsnit 5)
3. **Træningsloopet** — vi træner et netværk til at spotte legendariske Pokémon (afsnit 6)

> **Om opgaverne:** Der er med vilje flere opgaver, end I kan nå — ingen forventes at nå alt. Opgaver mærket **(ekstra)** er til jer, der er foran; opgaver mærket **(find fejlen)** har en bevidst fejl, som I skal finde og rette (så en fejl dér er meningen).

## Setup

In [ ]:
# Henter data + plottehjælper fra GitHub (Plan B: upload dem manuelt via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/data/Pokemon.csv
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/hjaelpefunktioner.py

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

from hjaelpefunktioner import plot_beslutningsgraense

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("Pokemon.csv")
df.head(3)

# 4: Fra linje til neuron

En **neuron** (opkaldt efter hjernecellerne, meget løst) er bare en lille regnemaskine, der:

1. tager nogle input-tal $x_1, x_2, \dots$
2. ganger hvert input med sin egen **vægt** $w_1, w_2, \dots$ og lægger en **bias** $b$ til:
$$z = w_1 x_1 + w_2 x_2 + \dots + b$$
3. (og som regel: sender resultatet gennem en **aktiveringsfunktion** — dem dykker vi ned i i notebook 4)

Genkender I trin 2? **Det er jo bare linjens ligning med flere variable!** En neuron med ét
input er bogstaveligt talt $z = ax + b$. Så her er dagens plot-twist: `nn.Linear(1, 1)` fra
notebook 2 **er** en neuron. Regressionen, I lavede, var et neuralt netværk med én neuron.
Surprise — I har allerede trænet jeres første netværk.

`nn.Linear` starter med tilfældige vægte (derfor `torch.manual_seed`!), men vi kan også
sætte dem selv:

In [ ]:
neuron = nn.Linear(1, 1)     # 1 input → 1 output
print("tilfældig vægt:", neuron.weight.item())
print("tilfældig bias:", neuron.bias.item())

with torch.no_grad():        # vi piller ved vægtene manuelt — det skal autograd ikke se
    neuron.weight[0, 0] = 2.0
    neuron.bias[0] = 1.0

x = torch.tensor([[3.0]])    # NB: nn.Linear vil have 2D-input: (antal eksempler, antal features)
print("neuron(3) =", neuron(x).item(), "  (tjek: 2 · 3 + 1 = 7)")

## Mange neuroner = et lag

`nn.Linear(6, 4)` er **4 neuroner**, der hver især kigger på de samme 6 input og laver
deres egen vægtede sum. Vægtene samles i en matrix med shape `(4, 6)` — én række pr.
neuron, én kolonne pr. input:

In [ ]:
lag = nn.Linear(6, 4)
print("vægte:", lag.weight.shape)   # (4 neuroner, 6 input)
print("bias:  ", lag.bias.shape)    # én bias pr. neuron

otte_pokemon = torch.randn(8, 6)    # 8 tilfældige "Pokémon" med 6 stats
print("output:", lag(otte_pokemon).shape)   # 8 eksempler ind → 8 gange 4 tal ud

## Krølningen: en hurtig smagsprøve på aktiveringsfunktioner

Stabler man to `nn.Linear`-lag direkte oven på hinanden, får man... stadig bare en lineær
funktion (mere om hvorfor i opgave 4.6). Tricket, der giver netværk deres kraft, er at
putte en **ikke-lineær funktion** ind mellem lagene.

Den mest berømte er **sigmoid**: $\sigma(z) = \frac{1}{1+e^{-z}}$. Den maser ethvert tal
ind i intervallet $(0, 1)$ — store tal → tæt på 1, små (negative) tal → tæt på 0. Perfekt,
når svaret skal være en **sandsynlighed** (fx "hvor sikker er jeg på, at den her er
legendarisk?"):

In [ ]:
z = torch.tensor([-5.0, -1.0, 0.0, 1.0, 5.0])
print(torch.sigmoid(z))   # alt bliver klemt ind mellem 0 og 1

### Opgaver

##### Opgave 4.1
En neuron har vægtene $w_1 = 0{,}5$, $w_2 = -1$ og bias $b = 2$, og får input
$x_1 = 4$, $x_2 = 3$. Regn først neuronens output $z = w_1 x_1 + w_2 x_2 + b$ **i hånden**
— og udfyld så formlen i koden og tjek dit facit.

In [ ]:
w1, w2, b = 0.5, -1.0, 2.0
x1, x2 = 4.0, 3.0
z = w1 * x1 + w2 * x2 + b
print(z)   # 0.5·4 + (-1)·3 + 2 = 2 - 3 + 2 = 1

<span style="color:red"><b>LØSNINGSFORSLAG:</b> z = 1. Det er ALT, hvad en neuron gør — gange, lægge sammen, plus bias. Ingen magi, kun folkeskole-matematik i industrielle mængder.</span>

##### Opgave 4.2
Cellen laver `nn.Linear(3, 1)`. Ændr den til `nn.Linear(6, 4)` og derefter `nn.Linear(4, 1)`
— men **forudsig `weight.shape` FØR I kører** hver gang. Reglen: shape er
`(antal output, antal input)`.

In [ ]:
for ind, ud in [(3, 1), (6, 4), (4, 1)]:
    lag = nn.Linear(ind, ud)
    print(f"nn.Linear({ind}, {ud}): weight.shape = {lag.weight.shape}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> (1, 3), (4, 6) og (1, 4) — altid (output, input). Antal vægte i et lag er produktet, plus én bias pr. output-neuron: nn.Linear(6, 4) har 6·4 + 4 = 28 parametre.</span>

##### Opgave 4.3
I notebook 2 (opgave 7.7) fandt gradient descent linjen $y \approx 0{,}74 \cdot x + 0$ for
Attack → Total. Sæt neuronens vægt og bias til de tal og tegn dens forudsigelser oven på
punktskyen — tegner én neuron præcis den samme linje som regressionen?

In [ ]:
x_attack = torch.tensor(df["Attack"].values, dtype=torch.float32)
y_total = torch.tensor(df["Total"].values, dtype=torch.float32)
x_attack = (x_attack - x_attack.mean()) / x_attack.std()
y_total = (y_total - y_total.mean()) / y_total.std()

neuron = nn.Linear(1, 1)
with torch.no_grad():
    neuron.weight[0, 0] = 0.74
    neuron.bias[0] = 0.0

with torch.no_grad():
    y_hat = neuron(x_attack.reshape(-1, 1)).squeeze()

plt.scatter(x_attack, y_total, alpha=0.3)
plt.plot(x_attack, y_hat, color="crimson", linewidth=2)
plt.xlabel("Attack (std)")
plt.ylabel("Total (std)")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Ja — nøjagtig samme linje. Én neuron uden aktiveringsfunktion ER lineær regression. Neurale netværk er "bare" mange af dem, der samarbejder.</span>

##### Opgave 4.4 (find fejlen)
Cellen fodrer et lag med Pokémon-data, men PyTorch protesterer højlydt. Læs fejlbeskeden —
den fortæller, hvilke shapes der støder sammen — og ret **laget** (ikke dataene), så det
passer til input med 5 features.

In [ ]:
fem_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def"]
X5 = torch.tensor(df[fem_stats].values, dtype=torch.float32)
print("input shape:", X5.shape)

lag = nn.Linear(5, 1)   # ← lagets input-antal skal matche antallet af features
print(lag(X5).shape)

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Fejlen ("mat1 and mat2 shapes cannot be multiplied (800x5 and 6x1)") siger præcis problemet: laget forventede 6 input pr. eksempel, men fik 5. Antal features i data og lagets første tal SKAL være ens — den regel kommer igen i opgave 5.5 om at stable lag.</span>

##### Opgave 4.5
Neuronen nedenfor har sigmoid på outputtet. Kør cellen, og fjern så `torch.sigmoid(...)`
(behold bare `z`) og kør igen. Sammenlign de to output-intervaller — hvornår ville man
ønske sig hvilket?

In [ ]:
neuron = nn.Linear(6, 1)
seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X = torch.tensor(df[seks_stats].values, dtype=torch.float32)
X = (X - X.mean(dim=0)) / X.std(dim=0)

with torch.no_grad():
    z = neuron(X).squeeze()
print("uden sigmoid:", z.min().item(), "til", z.max().item())
print("med sigmoid: ", torch.sigmoid(z).min().item(), "til", torch.sigmoid(z).max().item())

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Uden sigmoid kan outputtet være et hvilket som helst tal (positivt som negativt) — det vil man have til <b>regression</b> (forudsig HP, pris, temperatur...). Med sigmoid ligger alt i (0, 1) — det vil man have, når svaret er en <b>sandsynlighed</b> (klassifikation). Valget af "output-aktivering" afhænger altså af opgaven — det bliver et hovedtema i notebook 4.</span>

##### Opgave 4.6 (ekstra)
Tag to lineære funktioner, fx $f(x) = 2x + 1$ og $g(x) = 3x - 2$, og sæt dem sammen:
regn $g(f(x))$ ud på papir. Hvilken slags funktion får I? Hvad fortæller det om at stable
`nn.Linear`-lag **uden** noget imellem?

*Skriv jeres svar her:* $\dots$

<span style="color:red"><b>LØSNINGSFORSLAG:</b> g(f(x)) = 3(2x + 1) − 2 = 6x + 1 — bare endnu en lineær funktion! Lineært efter lineært giver altid lineært, uanset hvor mange lag man stabler. Derfor SKAL der en ikke-lineær aktiveringsfunktion ind mellem lagene — ellers er et 100-lags netværk matematisk set stadig kun én ret linje. Det eksperiment laver vi live i notebook 4.</span>

##### Opgave 4.7
Udfyld tallene, så laget tager **6 stats ind** og giver **3 neuroner** ud. Tjek formen bagefter.

In [ ]:
lag = nn.Linear(6, 3)
print(lag.weight.shape)        # forventet: (3, 6) — 3 neuroner, hver med 6 vægte

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>nn.Linear(6, 3)</code>. Vægt-matricen har form (ud, ind) = (3, 6): én række pr. neuron, én søjle pr. input.</span>

##### Opgave 4.8 (find fejlen)
Cellen skal sende tre tal gennem en neuron, men den crasher med en fejl om, at former
ikke passer sammen. Find og ret det ene tal, der er galt.

In [ ]:
neuron = nn.Linear(3, 1)                 # RETTET: 3 tal ind
x = torch.tensor([1.0, 2.0, 3.0])
print(neuron(x))

<span style="color:red"><b>LØSNINGSFORSLAG:</b> En neuron med <code>nn.Linear(1, 1)</code> kan kun gange med ét input. Med tre tal ind skal det være <code>nn.Linear(3, 1)</code> — antal input SKAL matche laget.</span>

##### Opgave 4.9 (ekstra)
Byg et bittelille netværk i hånden: send en Pokémons seks stats gennem et lag og krøl
resultatet med sigmoid. Udfyld kaldet, der sender `x` gennem laget.

In [ ]:
seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
x = torch.tensor(df[seks_stats].iloc[0].values, dtype=torch.float32)
lag = nn.Linear(6, 4)
skjult = torch.sigmoid(lag(x))
print("fire skjulte neuroner:", skjult)

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>lag(x)</code> giver fire tal (ét pr. neuron), og <code>torch.sigmoid</code> klemmer hvert af dem ind mellem 0 og 1. Det er præcis ét lag + én aktivering — byggestenen i ethvert neuralt netværk.</span>

# 5: `nn.Module` — netværk er klasser

Nu skal lagene stables til et rigtigt netværk — og her kommer jeres **klasse**-viden fra
intro-programmering i spil, for i PyTorch skriver man netværk som klasser. Kort recap:

- `class MinKlasse:` definerer en skabelon med **attributter** (data) og **metoder** (funktioner).
- `__init__(self)` køres, når man opretter et objekt — her gemmer man attributterne på `self`.
- `class MinKlasse(AndenKlasse):` betyder **nedarvning**: klassen får alt, hvad `AndenKlasse` kan, gratis.

(Rusten? Kig tilbage i afsnit P13 i [Intro-Programmering, notebook 3](../Intro-Programmering/3-Plots-klasser-og-tekst.ipynb).)

Et PyTorch-netværk er en klasse, der nedarver fra `nn.Module` og udfylder to ting:

1. **`__init__`**: hvilke lag har netværket? (byggeklodserne)
2. **`forward`**: i hvilken rækkefølge skal data flyde igennem dem? (samlevejledningen)

Mellem lagene bruger vi aktiveringsfunktionen **ReLU** — den er endnu simplere end sigmoid:
negative tal bliver til 0, positive tal passerer urørt. Mere om *hvorfor* i notebook 4;
indtil videre er ReLU bare vores "krølning" mellem lagene.

In [ ]:
class MitNetvaerk(nn.Module):
    def __init__(self):
        super().__init__()                  # tænd for alt nn.Module-maskineriet — ALDRIG glemme!
        self.lag1 = nn.Linear(6, 16)        # 6 stats ind → 16 skjulte neuroner
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(16, 1)        # 16 skjulte → 1 output

    def forward(self, x):                   # sådan flyder data gennem netværket
        x = self.lag1(x)
        x = self.aktivering(x)
        x = self.lag2(x)
        return x

model = MitNetvaerk()
print(model)

De 16 neuroner i midten kaldes et **skjult lag** (skjult, fordi hverken input eller
output ser dem direkte). Antallet — 16 — er et frit valg: flere neuroner = klogere netværk,
men også flere parametre at træne.

Hvor mange parametre (vægte + bias'er) har vores netværk egentlig? `p.numel()` tæller
tallene i én parameter-tensor, og `model.parameters()` går alle igennem:

In [ ]:
antal = sum(p.numel() for p in model.parameters())
print("antal parametre:", antal)
# tjek i hånden: lag1: 6·16 + 16 = 112, lag2: 16·1 + 1 = 17 → 129 i alt

Og netværket kan bruges med det samme — kald det som en funktion (det kalder `forward`
bag kulisserne). Uden træning er outputtet naturligvis rent nonsens fra tilfældige vægte:

In [ ]:
seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X = torch.tensor(df[seks_stats].values, dtype=torch.float32)
X = (X - X.mean(dim=0)) / X.std(dim=0)      # standardisering — I kender rytmen

with torch.no_grad():
    print(model(X[:5]).squeeze())            # 5 utrænede "gæt" — nonsens, men FORMEN er rigtig

### Opgaver

##### Opgave 5.1
Netværket ovenfor har 16 skjulte neuroner. Lav to nye udgaver: én med **4** og én med
**64** skjulte neuroner (ret begge `nn.Linear`-linjer, så dimensionerne stadig passer!),
og print antal parametre for hver. Hvor meget voksede netværket?

In [ ]:
for skjulte in [4, 16, 64]:
    class Netvaerk(nn.Module):
        def __init__(self):
            super().__init__()
            self.lag1 = nn.Linear(6, skjulte)
            self.aktivering = nn.ReLU()
            self.lag2 = nn.Linear(skjulte, 1)

        def forward(self, x):
            return self.lag2(self.aktivering(self.lag1(x)))

    print(f"{skjulte} skjulte neuroner: {sum(p.numel() for p in Netvaerk().parameters())} parametre")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> 4 skjulte: 33 parametre. 16: 129. 64: 513. Parametertallet vokser (stort set) lineært med antal skjulte neuroner her — men stabler man flere STORE lag oven på hinanden, eksploderer det hurtigt. GPT-modeller er præcis det her princip skruet op til flere hundrede milliarder parametre.</span>

##### Opgave 5.2
Udfyld `forward`-metoden: data skal flyde **lag1 → aktivering → lag2**.

In [ ]:
class Netvaerk(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(6, 16)
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(16, 1)

    def forward(self, x):
        x = self.lag1(x)
        x = self.aktivering(x)
        x = self.lag2(x)
        return x

model = Netvaerk()
with torch.no_grad():
    print(model(X[:3]).squeeze())

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Rækkefølgen i <code>forward</code> ER netværket — lagene i <code>__init__</code> er bare byggeklodser på hylden, og <code>forward</code> bestemmer, hvordan de sættes sammen.</span>

##### Opgave 5.3 (find fejlen)
Klassen nedenfor crasher allerede, når man prøver at **oprette** modellen. Kør cellen, læs
fejlbeskeden, og brug jeres klasse-viden: hvad er det for en vigtig linje, der mangler i
`__init__`?

In [ ]:
class OedelagtNetvaerk(nn.Module):
    def __init__(self):
        super().__init__()   # ← den manglende linje!
        self.lag1 = nn.Linear(6, 16)
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(16, 1)

    def forward(self, x):
        return self.lag2(self.aktivering(self.lag1(x)))

model = OedelagtNetvaerk()
print("nu virker det!")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>super().__init__()</code> kalder nn.Modules egen <code>__init__</code>, som sætter alt bogholderiet op (registrering af lag, parametre, osv.). Uden den er "hylderne ikke sat op", når man prøver at stille lag på dem — deraf fejlen. Reglen gælder generelt ved nedarvning: vil man have forældreklassens opsætning, skal man selv kalde den.</span>

##### Opgave 5.4 (find fejlen)
Denne klasse kan godt oprettes — men når man KALDER modellen, brokker PyTorch sig over, at
`forward` ikke er implementeret. Men den står der da...?! Kig meget nøje på metodens navn.

In [ ]:
class MystiskNetvaerk(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(6, 16)
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(16, 1)

    def forward(self, x):   # ← lille f! Python er case-sensitive
        return self.lag2(self.aktivering(self.lag1(x)))

model = MystiskNetvaerk()
with torch.no_grad():
    print(model(X[:3]).squeeze())

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Metoden hed <code>Forward</code> med stort F. Python ser det bare som en helt ny, ubrugt metode — sproget kan ikke vide, at man MENTE <code>forward</code>. Når man kalder <code>model(x)</code>, leder nn.Module efter en metode, der hedder præcis <code>forward</code>, finder kun sin egen tomme udgave, og kaster fejlen. Stavning og store/små bogstaver betyder ALT ved nedarvning.</span>

##### Opgave 5.5
Tilføj et **ekstra skjult lag** til netværket, så data flyder 6 → 16 → 8 → 1. Husk reglen
fra opgave 4.4: hvert lags input-antal skal matche det forrige lags output-antal — som
togskinner, der skal passe sammen.

In [ ]:
class Netvaerk(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(6, 16)
        self.lag2 = nn.Linear(16, 8)
        self.lag3 = nn.Linear(8, 1)
        self.aktivering = nn.ReLU()

    def forward(self, x):
        x = self.aktivering(self.lag1(x))
        x = self.aktivering(self.lag2(x))
        x = self.lag3(x)
        return x

model = Netvaerk()
print(model)

<span style="color:red"><b>LØSNINGSFORSLAG:</b> 6 → 16, 16 → 8, 8 → 1 — output-tallet i ét lag er input-tallet i det næste. Bemærk at aktiveringen bruges efter HVERT skjult lag (samme ReLU-klods kan genbruges — den har ingen parametre), men IKKE efter det sidste lag.</span>

##### Opgave 5.6 (ekstra)
Byg et netværk med arkitekturen **6 → 32 → 16 → 1** ved at udfylde alle dimensionerne.

In [ ]:
class StortNetvaerk(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(6, 32)
        self.lag2 = nn.Linear(32, 16)
        self.lag3 = nn.Linear(16, 1)
        self.aktivering = nn.ReLU()

    def forward(self, x):
        x = self.aktivering(self.lag1(x))
        x = self.aktivering(self.lag2(x))
        x = self.lag3(x)
        return x

model = StortNetvaerk()
print(model)
print("parametre:", sum(p.numel() for p in model.parameters()))

<span style="color:red"><b>LØSNINGSFORSLAG:</b> (6, 32), (32, 16), (16, 1) — i alt 6·32+32 + 32·16+16 + 16·1+1 = 769 parametre. Gradient descent skal altså finde 769 tal på én gang — og autograd leverer alle 769 gradienter med ét <code>backward()</code>-kald.</span>

##### Opgave 5.7
Kør cellen nedenfor 3-4 gange. Den opretter et NYT netværk hver gang og lader det gætte på
de samme 3 Pokémon — men gættene er nye hver gang! Hvorfor? Og hvorfor gav vores celler
tidligere i notebooken samme resultat hver gang, selvom vægtene var "tilfældige"?

In [ ]:
nyt_netvaerk = MitNetvaerk()
with torch.no_grad():
    print(nyt_netvaerk(X[:3]).squeeze())

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Hvert nyt netværk fødes med nye tilfældige vægte, så de utrænede gæt bliver forskellige. Tidligere i notebooken kørte vi <code>torch.manual_seed(42)</code> i setup-cellen — det låser den tilfældige talgenerator, så alle får SAMME "tilfældige" tal i samme rækkefølge (genkører man HELE notebooken fra toppen, får man derfor identiske resultater). I denne celle opretter vi bare endnu et netværk længere fremme i talrækken — deraf nye tal for hvert klik.</span>

##### Opgave 5.8
Kig på `MitNetvaerk`-klassen. Hvad har **vi selv** skrevet, og hvad **arver** vi gratis fra
`nn.Module`? (Tip: vi har fx aldrig skrevet koden, der gør, at `model(X)` virker, eller at
`model.parameters()` kan finde alle vægtene...)

*Skriv jeres svar her:* $\dots$

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Vi har kun skrevet: hvilke lag der findes (<code>__init__</code>) og rækkefølgen (<code>forward</code>). Fra nn.Module arver vi alt det hårde: at <code>model(X)</code> automatisk kalder forward, at <code>parameters()</code> finder alle vægte (så optimizeren kan opdatere dem), print-formatet, muligheden for at gemme/indlæse modellen, og meget mere. Det er nedarvningens kerneidé: skriv kun det, der er SÆRLIGT for din klasse.</span>

# 6: Træningsloopet — er den Pokémon legendarisk?

Tid til at samle det hele! Missionen: træn et netværk, der ud fra de seks kamp-stats
afgør, om en Pokémon er **legendarisk**. Det er **binær klassifikation** — svaret er
0 (almindelig) eller 1 (legendarisk) — så opskriften er:

- Output-laget skal give **én** sandsynlighed → 1 output-neuron med **sigmoid** til sidst.
- Tabsfunktionen skal måle "hvor forkert er en sandsynlighed?" → **`nn.BCELoss`**
 (*binary cross entropy* — den straffer hårdt, når modellen er *selvsikker og forkert*).

## Trin 1: Train/test-split — eksamensreglen

Vi deler dataene i **træningssæt** (80 %) og **testsæt** (20 %). Modellen må KUN lære af
træningssættet — testsættet er gemt væk til den afsluttende eksamen. At måle på data,
modellen har trænet på, svarer til at give eleverne facitlisten med til eksamen: flotte
karakterer, nul viden.

In [ ]:
seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X_np = df[seks_stats].values.astype("float32")
X_np = (X_np - X_np.mean(axis=0)) / X_np.std(axis=0)          # standardisering, som altid
y_np = df["Legendary"].astype(int).values.astype("float32")   # True/False → 1.0/0.0

X_traen, X_test, y_traen, y_test = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

X_traen = torch.tensor(X_traen)
X_test = torch.tensor(X_test)
y_traen = torch.tensor(y_traen)
y_test = torch.tensor(y_test)

print("træning:", X_traen.shape, "| test:", X_test.shape)
print("andel legendariske i træningssættet:", round(y_traen.mean().item(), 3))

## Trin 2: Modellen

6 stats ind → 16 skjulte neuroner (ReLU) → 1 sandsynlighed ud (sigmoid):

In [ ]:
class LegendeSpotter(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(6, 16)
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.lag1(x)
        x = self.aktivering(x)
        x = self.lag2(x)
        x = self.sigmoid(x)      # output bliver en sandsynlighed mellem 0 og 1
        return x

model = LegendeSpotter()
print(model)

## Trin 3: Træningsloopet

Præcis samme rytme som i notebook 2 — læg mærke til de fem nummererede trin, for de er
**altid** de samme. Ny ven: optimizeren **Adam**, en smartere udgave af SGD, der løbende
justerer skridtlængden selv. Den er standardvalget i praksis:

In [ ]:
model = LegendeSpotter()
tabsfunktion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
tab_historik = []

for epoke in range(500):
    # 1. Nulstil gradienter
    optimizer.zero_grad()
    # 2. Forward: modellens gæt for alle trænings-Pokémon
    y_hat = model(X_traen).squeeze()
    # 3. Regn tabet: hvor forkerte er gættene?
    tab = tabsfunktion(y_hat, y_traen)
    # 4. Backward: find alle gradienter
    tab.backward()
    # 5. Tag ét skridt ned ad bakken
    optimizer.step()

    tab_historik.append(tab.item())   # .item(): træk TALLET ud af tensoren

print("sluttab:", round(tab_historik[-1], 4))

plt.plot(tab_historik)
plt.xlabel("epoke")
plt.ylabel("BCE-tab")
plt.title("Tabskurven — den skal falde og flade ud")
plt.show()

## Trin 4: Eksamen — hvor god er den på testsættet?

Modellen svarer med sandsynligheder. Vi runder af ved 0,5 ("tror du mest ja eller mest
nej?") og sammenligner med de rigtige svar. Andelen af rigtige gæt kaldes **accuracy** —
og husk tricket fra opgave 3.8: gennemsnittet af en True/False-række er netop andelen af True:

In [ ]:
with torch.no_grad():
    sandsynligheder = model(X_test).squeeze()

gaet = (sandsynligheder > 0.5).float()          # sandsynlighed → 0 eller 1
accuracy = (gaet == y_test).float().mean()
print(f"test-accuracy: {accuracy.item():.1%}")

Pæn høj accuracy!...eller hvad?

## Fælden: den dovne model

Kun ca. 8 % af alle Pokémon er legendariske. En "model", der ALTID svarer "ikke
legendarisk" — uden at kigge på så meget som én stat — ville altså ramme rigtigt ca. 92 %
af gangene. **Høj accuracy er billig, når klasserne er skæve!** Det tjekker vi i opgave
10.6, og i opgave 6.5-10.8 finder vi ud af, hvad modellen egentlig har lært.

### Opgaver

##### Opgave 6.1
Skabelonen nedenfor er træningsloopet med fem huller — ét pr. nummereret trin. Udfyld dem
(kig IKKE i cellen ovenfor... okay, kig lidt hvis I sidder fast ).

In [ ]:
model_a = LegendeSpotter()
tabsfunktion = nn.BCELoss()
optimizer = torch.optim.Adam(model_a.parameters(), lr=0.01)

for epoke in range(500):
    # 1. Nulstil gradienter
    optimizer.zero_grad()
    # 2. Forward: modellens gæt
    y_hat = model_a(X_traen).squeeze()
    # 3. Regn tabet
    tab = tabsfunktion(y_hat, y_traen)
    # 4. Backward: find gradienterne
    tab.backward()
    # 5. Tag ét skridt
    optimizer.step()

print("sluttab:", round(tab.item(), 4))

<span style="color:red"><b>LØSNINGSFORSLAG:</b> De fem trin — nulstil, forward, tab, backward, step — er DNA'et i al PyTorch-træning. I møder præcis samme skelet igen i notebook 4, i notebook 5 og i resten af campen.</span>

##### Opgave 6.2
Skru på `lr` og antal epoker i træningsloopet, og mål test-accuracy hver gang (genbrug
eksamens-cellen). Hvor højt kan I komme — kan I slå 95 %? Notér jeres bedste resultat, og
kør jeres bedste opskrift to gange: får I samme tal?

In [ ]:
model_b = LegendeSpotter()
tabsfunktion = nn.BCELoss()
optimizer = torch.optim.Adam(model_b.parameters(), lr=0.005)
for epoke in range(2000):
    optimizer.zero_grad()
    y_hat = model_b(X_traen).squeeze()
    tab = tabsfunktion(y_hat, y_traen)
    tab.backward()
    optimizer.step()

with torch.no_grad():
    gaet = (model_b(X_test).squeeze() > 0.5).float()
print(f"test-accuracy: {(gaet == y_test).float().mean().item():.1%}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Realistiske resultater ligger typisk mellem 92 og 96 % — og de svinger fra kørsel til kørsel, fordi startvægtene er tilfældige (nyt netværk = nyt lotteri, jf. 9.7). At 95 %+ kræver lidt held, er en vigtig lærdom i sig selv. Og bemærk: bare at skrue epoker HELT op giver ikke evig fremgang — på et tidspunkt lærer modellen træningssættet udenad (overfitting — smugkig på 10.11 og notebook 5). Den STORE øjenåbner om, hvad de her procenter overhovedet er værd, venter i opgave 6.6...</span>

##### Opgave 6.3 (find fejlen)
Loopet nedenfor kører uden fejlbeskeder... men tabet rokker sig ikke ud af stedet, og
accuracy er elendig. Rytmen er kommet i uorden: kig på rækkefølgen af de fem trin og ret
den. (Hvad SKAL der være sket, før `step()` kan tage et fornuftigt skridt? Og hvornår må
man tidligst nulstille?)

In [ ]:
model_c = LegendeSpotter()
tabsfunktion = nn.BCELoss()
optimizer = torch.optim.Adam(model_c.parameters(), lr=0.01)

for epoke in range(500):
    optimizer.zero_grad()     # 1. nulstil FØRST
    y_hat = model_c(X_traen).squeeze()
    tab = tabsfunktion(y_hat, y_traen)
    tab.backward()            # 4. regn gradienter...
    optimizer.step()          # 5. ...og tag SÅ skridtet

print("sluttab:", round(tab.item(), 4))

<span style="color:red"><b>LØSNINGSFORSLAG:</b> I den ødelagte version tager <code>step()</code> sit skridt FØR <code>backward()</code> har regnet gradienterne (så der er intet at gå efter), og <code>zero_grad()</code> sletter derefter de friske gradienter, inden næste runde bruger dem. Rytmen er ufravigelig: nulstil → forward → tab → backward → step. Læg mærke til, at fejlen IKKE gav nogen fejlbesked — den slags "stille fejl" er de farligste i ML.</span>

##### Opgave 6.4 (find fejlen)
Nogen ville gemme tabskurven, men plottet crasher med en fejl om "requires grad". Kig på
linjen med `append` — hvad mangler der? (Tip: hvad gør `.item()`, og hvorfor brugte vi den
i det rigtige loop?)

In [ ]:
model_d = LegendeSpotter()
tabsfunktion = nn.BCELoss()
optimizer = torch.optim.Adam(model_d.parameters(), lr=0.01)
tab_historik = []

for epoke in range(200):
    optimizer.zero_grad()
    y_hat = model_d(X_traen).squeeze()
    tab = tabsfunktion(y_hat, y_traen)
    tab.backward()
    optimizer.step()
    tab_historik.append(tab.item())   # ← .item() trækker det rene TAL ud af tensoren

plt.plot(tab_historik)
plt.xlabel("epoke")
plt.ylabel("tab")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>tab</code> er en tensor, der stadig hænger fast i hele autograd-regnenetværket — matplotlib kan (og skal) ikke plotte den slags. <code>.item()</code> klipper navlestrengen og giver et almindeligt Python-tal. (Bonus: uden <code>.item()</code> holder listen også hele regnehistorikken i live og æder hukommelse.)</span>

##### Opgave 6.5
Træn tre modeller med læringsraterne **0.001, 0.05 og 1.0** og plot deres tre tabskurver
i samme figur (skabelonen er klar — den mangler bare `label` og `plt.legend()`). Sæt ord
på hver kurve: tålmodig? effektiv? kaotisk?

In [ ]:
for lr in [0.001, 0.05, 1.0]:
    model_e = LegendeSpotter()
    tabsfunktion = nn.BCELoss()
    optimizer = torch.optim.Adam(model_e.parameters(), lr=lr)
    historik = []
    for epoke in range(300):
        optimizer.zero_grad()
        tab = tabsfunktion(model_e(X_traen).squeeze(), y_traen)
        tab.backward()
        optimizer.step()
        historik.append(tab.item())
    plt.plot(historik, label=f"lr = {lr}")

plt.legend()
plt.xlabel("epoke")
plt.ylabel("tab")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> lr = 0.001: falder pænt men langsomt (når ikke i mål på 300 epoker). lr = 0.05: falder hurtigt og fladt ud — fin balance. lr = 1.0: hopper kaotisk rundt og kan ende HØJERE, end den startede — for store skridt, præcis som i opgave 7.1. Samme historie som i notebook 2, bare med et rigtigt netværk.</span>

##### Opgave 6.6
Den dovne model: beregn accuracy for "modellen", der ALTID gætter "ikke legendarisk"
(altså gæt = 0 for alle), og sammenlign med jeres netværk. Slår jeres netværk overhovedet
den dovne? Og find så ud af det VIGTIGE tal: af de legendariske Pokémon i testsættet —
hvor mange fanger hver "model"?

In [ ]:
dovne_gaet = torch.zeros(len(y_test))
doven_accuracy = (dovne_gaet == y_test).float().mean()
print(f"den dovne models accuracy: {doven_accuracy.item():.1%}")

with torch.no_grad():
    gaet = (model(X_test).squeeze() > 0.5).float()
print(f"netværkets accuracy:       {(gaet == y_test).float().mean().item():.1%}")

legendariske = y_test == 1
print(f"legendariske i testsættet: {int(legendariske.sum())}")
print(f"dovne model fanger:        0")
print(f"netværket fanger:          {int(gaet[legendariske].sum())}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Den dovne model rammer ca. 94 % på vores testsæt (der tilfældigvis kun har 10 legendariske ud af 160) — men fanger PRÆCIS NUL legendariske. Netværket ligger typisk kun et par procentpoint højere i accuracy... men fanger 5-8 af de 10! Moralen: ved skæve klasser skal man altid spørge "hvor mange af de SJÆLDNE fanger du?" — ikke kun "hvor tit har du ret?". Accuracy-forskellen mellem doven og god kan være bittesmå tal, mens forskellen i det, der BETYDER noget, er enorm. (Målene hedder recall/precision, og dem støder I på igen senere på campen.)</span>

##### Opgave 6.7
Udfyld eksamens-cellen: sandsynligheder → 0/1-gæt (grænse 0,5) → accuracy.

In [ ]:
with torch.no_grad():
    sandsynligheder = model(X_test).squeeze()

gaet = (sandsynligheder > 0.5).float()
accuracy = (gaet == y_test).float().mean()
print(f"test-accuracy: {accuracy.item():.1%}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Grænsen 0,5 betyder "gæt på det, du tror mest på". Man kan flytte grænsen: sænker man den (fx 0,3), fanger man flere legendariske, men får også flere falske alarmer — en skydeskive-indstilling, som afhænger af, hvad fejlene koster.</span>

##### Opgave 6.8
Hvor meget kan netværket nå med kun **2** features? Træn en model, der kun ser `HP` og
`Defense` (skabelonen har allerede skåret X ned — kolonne 0 og 2). Hvad sker der med
accuracy — og vigtigere: hvor mange legendariske fanger den? Prøv bagefter med et andet
feature-par og se, hvor meget PARRET betyder.

In [ ]:
X_traen2 = X_traen[:, [0, 2]]    # HP og Defense
X_test2 = X_test[:, [0, 2]]

class LilleSpotter(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(2, 16)
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.lag2(self.aktivering(self.lag1(x))))

model2 = LilleSpotter()
tabsfunktion = nn.BCELoss()
optimizer = torch.optim.Adam(model2.parameters(), lr=0.01)
for epoke in range(500):
    optimizer.zero_grad()
    tab = tabsfunktion(model2(X_traen2).squeeze(), y_traen)
    tab.backward()
    optimizer.step()

with torch.no_grad():
    gaet = (model2(X_test2).squeeze() > 0.5).float()
print(f"accuracy med 2 features: {(gaet == y_test).float().mean().item():.1%}")
print("fangede legendariske:", int(gaet[y_test == 1].sum()), "af", int((y_test == 1).sum()))

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Med HP + Defense falder accuracy til ca. den dovne models niveau, og modellen fanger typisk kun 0-2 af de 10 legendariske — de to stats indeholder simpelthen for lidt "legendariskhed". Men her kommer twisten: prøv <code>[1, 3]</code> (Attack + Sp. Atk) — det par alene fanger typisk ~7 af 10 og matcher eller SLÅR 6-feature-modellen! Featurevalg er halvdelen af faget: to stærke features kan slå seks blandede, fordi svage features mest bidrager med støj.</span>

##### Opgave 6.9 (ekstra)
To identiske netværk, to identiske loops — men det ene træner på de RÅ stats (uden
standardisering) med SGD, det andet på de standardiserede. Kør cellen og sammenlign.
Payoff for notebook 1! (Bemærk: vi bruger SGD her; prøv bagefter med Adam og se, at den
delvist redder situationen — men stadig ikke helt.)

In [ ]:
X_raa = torch.tensor(df[seks_stats].values.astype("float32"))
X_raa_traen, X_raa_test, _, _ = train_test_split(X_raa.numpy(), y_np, test_size=0.2, random_state=42)
X_raa_traen = torch.tensor(X_raa_traen)

for navn, data in [("standardiseret", X_traen), ("rå tal", X_raa_traen)]:
    torch.manual_seed(0)
    model_t = LegendeSpotter()
    tabsfunktion = nn.BCELoss()
    optimizer = torch.optim.SGD(model_t.parameters(), lr=0.1)
    for epoke in range(500):
        optimizer.zero_grad()
        tab = tabsfunktion(model_t(data).squeeze(), y_traen)
        tab.backward()
        optimizer.step()
    print(f"{navn}: sluttab = {round(tab.item(), 4)}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Med de standardiserede data falder tabet pænt; med de rå tal sidder tabet fast (eller hopper rundt) — sigmoid-neuronerne bliver "mættet" af de store input, gradienterne bliver mikroskopiske, og SGD kommer ingen vegne. Adam redder noget med sin adaptive skridtlængde, men den standardiserede version vinder stadig. Standardisering er ikke pynt — den kan være forskellen på en model, der lærer, og en, der aldrig kommer i gang.</span>

##### Opgave 6.10 (ekstra)
Byg klassifikationsopgaven om til **regression**: forudsig `HP` ud fra de fem ANDRE stats.
Tre ting skal ændres i forhold til LegendeSpotter-opskriften: (1) output-aktiveringen
(sigmoid?! til HP-værdier på 20-255?), (2) tabsfunktionen, (3) målingen til sidst (accuracy
giver ikke mening for kommatal — brug fx den gennemsnitlige fejl i HP-point).

In [ ]:
fem = ["Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X_hp = df[fem].values.astype("float32")
X_hp = (X_hp - X_hp.mean(axis=0)) / X_hp.std(axis=0)
y_hp = df["HP"].values.astype("float32")

X_hp_traen, X_hp_test, y_hp_traen, y_hp_test = train_test_split(X_hp, y_hp, test_size=0.2, random_state=42)
X_hp_traen, X_hp_test = torch.tensor(X_hp_traen), torch.tensor(X_hp_test)
y_hp_traen, y_hp_test = torch.tensor(y_hp_traen), torch.tensor(y_hp_test)

class HPGaetter(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(5, 16)
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(16, 1)   # (1) INGEN sigmoid — output skal kunne være fx 140

    def forward(self, x):
        return self.lag2(self.aktivering(self.lag1(x)))

model_hp = HPGaetter()
tabsfunktion = nn.MSELoss()            # (2) MSE til regression
optimizer = torch.optim.Adam(model_hp.parameters(), lr=0.01)
for epoke in range(2000):
    optimizer.zero_grad()
    tab = tabsfunktion(model_hp(X_hp_traen).squeeze(), y_hp_traen)
    tab.backward()
    optimizer.step()

with torch.no_grad():                  # (3) gennemsnitlig fejl i HP-point
    fejl = (model_hp(X_hp_test).squeeze() - y_hp_test).abs().mean()
print(f"gennemsnitlig fejl: {fejl.item():.1f} HP-point")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> De tre ændringer: sigmoid fjernes (regression skal kunne ramme alle værdier), BCELoss → MSELoss, og accuracy → gennemsnitlig absolut fejl (her typisk ~15 HP-point). Bemærk at vi her standardiserede X men beholdt y i rå HP-point, så fejlen er til at fortolke — man kan også standardisere y (og regne tilbage), hvilket ofte træner endnu pænere.</span>

##### Opgave 6.11
Hvorfor snyder man sig selv, hvis man måler accuracy på **træningsdataene**? Og hvad tror
I, der sker med (a) accuracy på træningssættet og (b) accuracy på testsættet, hvis man
lader et STORT netværk træne i ekstremt mange epoker på et LILLE datasæt?

*Skriv jeres svar her:* $\dots$

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Modellen har set træningsdataene under træningen — den kan i værste fald have lært dem UDENAD (som at bedømme en elev på opgaver, eleven har fået facit til). Med stort netværk + få data + mange epoker går trænings-accuracy mod 100 %, mens test-accuracy stagnerer eller FALDER: modellen memorerer i stedet for at generalisere. Det hedder <b>overfitting</b>, og I skal se det ske med egne øjne i notebook 5 (opgave 14.6).</span>

# Godt gået!

I har bygget og trænet et rigtigt neuralt netværk med klasser, ReLU, BCELoss og Adam — og
I har lært at være sunde skeptikere over for høje accuracy-tal.

**Næste stop:** `4-Aktiveringsfunktioner.ipynb`. ReLU og sigmoid har indtil videre bare
været "krølning" — nu skal I se PRÆCIS hvad de gør, møde hele familien (Leaky ReLU, Tanh,
Softmax), og se med egne øjne, hvad der sker med et netværk, der ingen aktivering har.